# 3. Chunking y Embeddings con LangChain y Google AI Studio

## Objetivos de Aprendizaje
- Entender la necesidad de dividir documentos largos en fragmentos (chunking).
- Aprender a usar los divisores de texto de LangChain (`RecursiveCharacterTextSplitter`).
- Comprender qué son los embeddings y por qué son fundamentales para la búsqueda semántica.
- Generar embeddings para los chunks de texto usando **Google AI Studio** (capa gratuita: 1500 req/día).

## De la Búsqueda por Palabras Clave a la Búsqueda Semántica

En el notebook anterior, construimos un RAG básico que funcionaba con palabras clave. Sus limitaciones eran claras: no entendía el significado, los sinónimos o el contexto. Para superar esto, necesitamos dos procesos clave:

1.  **Chunking**: Si un documento es muy largo, su embedding representará una idea demasiado general. Al dividirlo en chunks más pequeños y cohesivos, cada chunk representa una idea más específica. Esto hace que la búsqueda de similitud sea mucho más precisa.
2.  **Embeddings**: Son la pieza central de la búsqueda semántica. Convierten el texto (nuestros chunks) en vectores numéricos en un espacio multidimensional. Los textos con significados similares tendrán vectores cercanos en este espacio, lo que nos permite encontrar información relevante aunque no se usen las mismas palabras.

## ⚠️ Nota importante: ¿Por qué no usamos OpenRouter?

OpenRouter **solo soporta el endpoint `/chat/completions`**. No expone el endpoint `/embeddings`, por lo que no podemos generar embeddings a través de él. Por eso, en este notebook usamos **Google AI Studio**, que sí ofrece embeddings gratuitos con un endpoint compatible con OpenAI.

In [ ]:
# Instalación de dependencias
!pip install openai python-dotenv langchain-text-splitters -q
print("✅ Dependencias instaladas")

In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import display, Markdown

# Cargar variables desde .env
load_dotenv()

# --- Configuración del cliente (Google AI Studio) ---
# Endpoint compatible con OpenAI para embeddings y chat
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
API_KEY = os.environ.get("GEMINI_API_KEY")

if not API_KEY:
    print("🔑 No se encontró GEMINI_API_KEY en el entorno.")
    API_KEY = getpass("Ingresa tu clave de Google AI Studio: ")
    os.environ["GEMINI_API_KEY"] = API_KEY

if not API_KEY or len(API_KEY) < 10:
    raise ValueError("❌ Clave inválida. Debe empezar por 'AIzaSy...'")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Modelo de embeddings de Google (gratuito)
EMBEDDING_MODEL = "text-embedding-004"

print("✅ Cliente inicializado correctamente.")
print(f"   Base URL: {BASE_URL}")
print(f"   Modelo de embeddings: {EMBEDDING_MODEL}")

## 1. El Documento a Procesar

Comenzamos con un documento de texto más largo que los que usamos en el notebook anterior. Este texto sobre la historia de la inteligencia artificial es ideal para demostrar la necesidad del chunking.

In [ ]:
long_text = (
    "La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. "
    "Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. "
    "El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia de Dartmouth, considerada el nacimiento oficial de la IA como campo de estudio. "
    "Los primeros años estuvieron marcados por un gran optimismo, con la creación de programas como el Logic Theorist y el General Problem Solver, que podían resolver problemas de lógica y teoremas matemáticos. "
    "Sin embargo, las limitaciones computacionales y la complejidad de los problemas del mundo real llevaron al primer 'invierno de la IA' en la década de 1970, un período de reducción de fondos y escepticismo. "
    "El resurgimiento llegó en la década de 1980 con el auge de los sistemas expertos, programas que encapsulaban el conocimiento de un experto humano en un dominio específico, como el diagnóstico médico (por ejemplo, MYCIN). "
    "Estos sistemas demostraron el valor comercial de la IA, pero su fragilidad y el alto costo de mantenimiento condujeron a un segundo invierno a finales de los 80 y principios de los 90. "
    "La revolución moderna de la IA comenzó a gestarse a finales de los 90 y principios de los 2000, impulsada por tres factores clave: la disponibilidad de grandes volúmenes de datos (Big Data), el desarrollo de hardware más potente (especialmente las GPU) y los avances en algoritmos de aprendizaje automático, en particular las redes neuronales profundas (deep learning). "
    "Hitos como la victoria de Deep Blue de IBM sobre el campeón de ajedrez Garry Kasparov en 1997 y, más tarde, el triunfo de AlphaGo de DeepMind en el juego de Go en 2016, demostraron el poder del aprendizaje por refuerzo y el deep learning. "
    "Hoy, vivimos en la era de los modelos de lenguaje grande (LLM) como GPT y Claude, y los modelos de difusión para la generación de imágenes, que han llevado la IA a la corriente principal, transformando industrias y planteando nuevas preguntas sobre el futuro de la tecnología y la humanidad."
)

display(Markdown("### 📜 Documento Original"))
print(long_text)
print(f"\nLongitud del texto: {len(long_text)} caracteres.")

## 2. Chunking con `RecursiveCharacterTextSplitter`

LangChain ofrece varias herramientas para dividir texto. `RecursiveCharacterTextSplitter` es una de las más recomendadas porque intenta dividir el texto basándose en una jerarquía de separadores (como saltos de línea dobles, saltos de línea simples, espacios, etc.) para mantener los fragmentos lo más coherentes posible.

-   `chunk_size`: Define el tamaño máximo de cada chunk (en caracteres).
-   `chunk_overlap`: Define cuántos caracteres se superponen entre chunks consecutivos. Esto es crucial para no perder el contexto que podría existir justo en el límite entre dos chunks.

In [ ]:
# Inicializar el divisor de texto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,      # Tamaño del chunk en caracteres
    chunk_overlap=50,    # Superposición para mantener contexto
    length_function=len
)

# Dividir el documento en chunks
chunks = text_splitter.split_text(long_text)

display(Markdown(f"### 🧩 Documento Dividido en {len(chunks)} Chunks"))

# Mostrar los chunks resultantes
for i, chunk in enumerate(chunks):
    print(f"--- CHUNK {i+1} (Longitud: {len(chunk)} caracteres) ---")
    print(chunk)
    print()

## 3. Generación de Embeddings para cada Chunk

Ahora que tenemos nuestros chunks, el siguiente paso es convertirlos en vectores numéricos.

Usaremos el modelo **`text-embedding-004`** de Google AI Studio, que es gratuito y tiene una dimensión de **768**. En OpenAI, el equivalente sería `text-embedding-3-small` (1536 dimensiones), pero aquí usamos Google porque **OpenRouter no soporta embeddings**.

In [ ]:
def get_embeddings(client, texts, model=EMBEDDING_MODEL):
    """
    Obtiene embeddings para una lista de textos.
    
    Args:
        client: Cliente de OpenAI (apuntando a Google AI Studio).
        texts (list): Lista de strings a vectorizar.
        model (str): Identificador del modelo de embeddings.
    
    Returns:
        list: Lista de vectores (cada uno es una lista de floats).
    """
    try:
        response = client.embeddings.create(
            model=model,
            input=texts
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print(f"❌ Error al generar embeddings: {e}")
        return []

# Generar embeddings para todos los chunks
chunk_embeddings = get_embeddings(client, chunks)

if chunk_embeddings:
    print(f"✓ Se generaron {len(chunk_embeddings)} embeddings.")
    print(f"Dimensión de cada embedding: {len(chunk_embeddings[0])}")
else:
    print("⚠️ No se pudieron generar embeddings. Revisa tu clave API.")

In [ ]:
# Mostrar un ejemplo de un chunk y su embedding correspondiente
if chunk_embeddings:
    display(Markdown("### 🔍 Ejemplo: Chunk y su Embedding"))
    
    example_index = 0
    example_chunk = chunks[example_index]
    example_embedding = chunk_embeddings[example_index]
    
    print(f"**Texto del Chunk {example_index + 1}:**\n{example_chunk}\n")
    print(f"**Embedding (primeros 10 de {len(example_embedding)} dimensiones):**")
    print(f"{example_embedding[:10]}...")

## 4. (Opcional) Alternativa: Embeddings Locales con `sentence-transformers`

Si no quieres depender de una API externa para embeddings, puedes generarlos **localmente** con la librería `sentence-transformers`. Esta opción:
- ✅ **No requiere clave API**.
- ✅ **Funciona offline** una vez descargado el modelo.
- ✅ **Completamente gratuita**.
- ❌ Es más lenta en CPU (aunque aceptable para documentos pequeños).
- ❌ El modelo ocupa ~120 MB en disco.

Esto es útil si:
- Tus estudiantes no tienen saldo en ningún proveedor.
- Quieres mostrar que los embeddings también pueden generarse localmente.
- Estás en un entorno con restricciones de red.

In [ ]:
# ⚠️ Descomenta y ejecuta esta celda si quieres probar la opción local
# !pip install sentence-transformers -q

# from sentence_transformers import SentenceTransformer

# # Modelo multilingüe (soporta español)
# modelo_local = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# # Generar embeddings localmente
# embeddings_locales = modelo_local.encode(chunks, show_progress_bar=True)

# print(f"✓ Se generaron {len(embeddings_locales)} embeddings localmente.")
# print(f"Dimensión de cada embedding: {len(embeddings_locales[0])}")
# print(f"Tipo: {type(embeddings_locales)}")

## 5. Conclusiones y Próximos Pasos

¡Felicidades! Has completado los dos pasos fundamentales para pasar de un RAG básico a uno semántico:

1.  **Has dividido un documento largo en chunks manejables y coherentes** usando `RecursiveCharacterTextSplitter`.
2.  **Has convertido cada chunk en un vector numérico (embedding)** que captura su significado, usando Google AI Studio.

Con esta lista de chunks y su correspondiente lista de embeddings, ahora tienes una **base de conocimiento vectorial** en memoria.

## 📊 Resumen de lo aprendido

| Concepto | Qué es | Por qué importa |
| :--- | :--- | :--- |
| **Chunking** | Dividir texto largo en fragmentos coherentes | Mejora la precisión del retrieval |
| **Overlap** | Solapamiento entre chunks | Evita perder contexto en los límites |
| **Embedding** | Vector numérico que representa el significado | Permite búsqueda semántica (no solo por palabras) |
| **Modelo de embeddings** | Red neuronal que genera los vectores | Determina la calidad de la búsqueda |

## 🔮 En el próximo notebook (`4-vector-rag.ipynb`), aprenderás a:

-   Tomar una consulta del usuario y generar también su embedding.
-   Usar la **similitud coseno** para comparar el vector de la consulta con todos los vectores de los chunks.
-   Seleccionar los chunks más similares (los que tienen la puntuación de similitud más alta) para usarlos como contexto en la generación de la respuesta.

## ⚠️ Recordatorio: OpenRouter NO sirve para embeddings

Si en el futuro quieres generar embeddings, **no uses OpenRouter**. Opciones recomendadas:
- **Google AI Studio** (gratis, 1500 req/día)
- **Hugging Face Inference API** (gratis, tasa limitada)
- **`sentence-transformers` local** (gratis, offline)
- **Jina AI** (gratis, 1M tokens/mes)

Para chat, OpenRouter sigue siendo excelente; pero para embeddings, usa otro proveedor.